# B prototype: evaluate
Development evidence only. Japan untouched.

In [ ]:
from pathlib import Path
Path('b_core.py').write_text('"""Source-only scale consistency for the pinned YOLO11 detector.\n\nNo target images, pseudo-labels, extra detection supervision, or architecture changes.\n"""\nimport json\nimport statistics\nimport time\nimport weakref\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\nfrom ultralytics.utils.tal import TaskAlignedAssigner,make_anchors,dist2bbox\n\n\ndef ramp(epoch,batch_index,batches):\n    return min(1.,max(0.,(epoch+batch_index/max(1,batches))/5.))\n\n\ndef geometry_selection(batch,batch_size,max_gt,focused):\n    """Use augmented normalized WH directly, avoiding xyxy cancellation."""\n    bidx=batch[\'batch_idx\'].long().view(-1)\n    sides=torch.round(batch[\'bboxes\'][:,2:4].double()*640*1e6)/1e6\n    visible=sides.min(-1).values>=8\n    small_thin=(sides.prod(-1)<1024)|(sides.min(-1).values<16)\n    chosen=visible&(small_thin if focused else torch.ones_like(visible))\n    counts=torch.bincount(bidx,minlength=batch_size)\n    offsets=torch.cat([counts.new_zeros(1),counts.cumsum(0)[:-1]])\n    local=torch.arange(len(bidx),device=bidx.device)-offsets[bidx]\n    out=torch.zeros((batch_size,max_gt),device=bidx.device,dtype=torch.bool)\n    if len(bidx): out[bidx,local]=chosen\n    return out,dict(total=int(len(bidx)),visible=int(visible.sum()),target=int((visible&small_thin).sum()),selected=int(chosen.sum()))\n\n\nclass RecordingAssigner(TaskAlignedAssigner):\n    def get_pos_mask(self,*args,**kwargs):\n        result=super().get_pos_mask(*args,**kwargs)\n        self.raw_alignment=result[1].detach().clone()\n        return result\n\n\ndef view_predictions(raw,batch,size,criterion):\n    """Differentiable decoded outputs; detached unmodified stock TAL matching."""\n    raw=criterion.parse_output(raw)\n    logits=raw[\'scores\'].permute(0,2,1).float().contiguous()\n    distributions=raw[\'boxes\'].permute(0,2,1).float().contiguous()\n    anchors,strides=make_anchors(raw[\'feats\'],criterion.stride,.5)\n    anchors=anchors.float(); strides=strides.float()\n    distances=distributions.view(*distributions.shape[:2],4,criterion.reg_max).softmax(-1)\n    distances=distances.matmul(criterion.proj.float())\n    boxes=dist2bbox(distances,anchors,xywh=False)*strides\n    probabilities=logits.sigmoid()\n    with torch.no_grad():\n        targets=torch.cat([batch[\'batch_idx\'].view(-1,1),batch[\'cls\'].view(-1,1),batch[\'bboxes\']],1).float()\n        targets=criterion.preprocess(targets,len(logits),torch.full((4,),size,device=logits.device,dtype=torch.float32))\n        labels,gt=targets.split((1,4),2); valid=gt.sum(-1,keepdim=True)>0\n        assigner=RecordingAssigner(topk=10,num_classes=criterion.nc,alpha=.5,beta=6.,stride=criterion.stride.tolist())\n        result=assigner(probabilities.detach(),boxes.detach(),anchors*strides,labels,gt,valid)\n        fg,indices=result[3:]\n        alignment=getattr(assigner,\'raw_alignment\',logits.new_zeros(len(logits),gt.shape[1],logits.shape[1]))\n    return dict(probabilities=probabilities,boxes=boxes/size,fg=fg,indices=indices,alignment=alignment,labels=labels.squeeze(-1).long(),valid=valid.squeeze(-1))\n\n\ndef top_three(view):\n    """Stable descending rank; equal scores retain ascending anchor order."""\n    p=view[\'probabilities\']; b,m,n=view[\'alignment\'].shape\n    if not m:\n        return p.new_zeros((b,0,p.shape[-1])),p.new_zeros((b,0,4)),torch.zeros((b,0),dtype=torch.bool,device=p.device),torch.zeros((b,0),dtype=torch.bool,device=p.device)\n    with torch.no_grad():\n        assigned=(view[\'indices\'][:,None,:]==torch.arange(m,device=p.device)[None,:,None])&view[\'fg\'][:,None,:]\n        ranked=view[\'alignment\'].masked_fill(~assigned,-1)\n        ids=torch.argsort(ranked,dim=-1,descending=True,stable=True)[...,:min(3,n)]\n        weights=view[\'alignment\'].gather(-1,ids)*assigned.gather(-1,ids)\n        mass=weights.sum(-1)\n        weights=weights/torch.where(mass>0,mass,torch.ones_like(mass))[...,None]\n        has_positive=assigned.any(-1)\n    batch_ids=torch.arange(b,device=p.device)[:,None,None]\n    return ((p[batch_ids,ids]*weights[...,None]).sum(-2),\n            (view[\'boxes\'][batch_ids,ids]*weights[...,None]).sum(-2),mass>0,has_positive)\n\n\ndef agreement(teacher_p,teacher_b,student_p,student_b):\n    p=teacher_p.detach().float().clamp(1e-6,1-1e-6)\n    q=student_p.float().clamp(1e-6,1-1e-6)\n    kl=(p*(p/q).log()+(1-p)*((1-p)/(1-q)).log()).mean(-1)\n    loc=(teacher_b.detach().float()-student_b.float()).abs().mean(-1)\n    return kl+loc,kl,loc\n\n\ndef consistency(student_raw,teacher_raw,batch,criterion,focused):\n    with torch.autocast(device_type=batch[\'img\'].device.type,enabled=False):\n        student=view_predictions(student_raw,batch,320,criterion)\n        with torch.no_grad(): teacher=view_predictions(teacher_raw,batch,640,criterion)\n        selected,stats=geometry_selection(batch,len(batch[\'img\']),student[\'labels\'].shape[1],focused)\n        sp,sb,sv,sh=top_three(student); tp,tb,tv,th=top_three(teacher)\n        usable=selected&sv&tv\n        losses,kl,loc=agreement(tp,tb,sp,sb)\n        zero=student[\'probabilities\'].sum()*0+student[\'boxes\'].sum()*0\n        count=usable.sum()\n        loss=(losses*usable).sum()/count.clamp_min(1)+zero\n        labels=student[\'labels\']\n        stats.update(usable=int(count),skip_no_positive=int((selected&(~sh|~th)).sum()),\n                     skip_zero_weight=int((selected&sh&th&(~sv|~tv)).sum()),\n                     selected_by_class=torch.bincount(labels[selected],minlength=4).tolist(),\n                     usable_by_class=torch.bincount(labels[usable],minlength=4).tolist(),\n                     kl=float(((kl*usable).sum()/count.clamp_min(1)).detach()),\n                     box_l1=float(((loc*usable).sum()/count.clamp_min(1)).detach()),aux=float(loss.detach()))\n        return loss,stats\n\n\nclass BContext:\n    def __init__(self,trainer,arm):\n        self.owner=weakref.ref(trainer); self.arm=arm; self.batch_index=0; self.last={}\n    def compute(self,model,batch):\n        trainer=self.owner(); teacher=trainer.ema.ema\n        assert not teacher.training and all(not p.requires_grad for p in teacher.parameters())\n        updates_before=trainer.ema.updates\n        half=F.interpolate(batch[\'img\'],size=(320,320),mode=\'bilinear\',align_corners=False,antialias=True)\n        half_raw=model(half)\n        with torch.no_grad(): teacher_raw=teacher(batch[\'img\'])\n        aux,stats=consistency(half_raw,teacher_raw,batch,model.criterion,self.arm==\'small_thin\')\n        weight=ramp(trainer.epoch,self.batch_index,len(trainer.train_loader))\n        stats.update(epoch=trainer.epoch+1,batch=self.batch_index,weight=weight,teacher_updates_before=updates_before)\n        self.batch_index+=1; self.last=stats\n        return aux*weight\n\n\nclass BDetectionModel(DetectionModel):\n    def loss(self,batch,preds=None):\n        loss,items=super().loss(batch,preds)\n        context=getattr(self,\'_b_context\',None)\n        if self.training and context is not None:\n            extra=context.compute(self,batch)*batch[\'img\'].shape[0]\n            # Stock returns a 3-component vector which the trainer sums. Add once.\n            loss=loss.clone(); loss[1]=loss[1]+extra\n        return loss,items\n\n\nclass BTrainer(DetectionTrainer):\n    """Stock training orchestration, optimizer and EMA; only model loss is extended."""\n    def __init__(self,*args,**kwargs):\n        self.b_config=json.loads(Path(\'b-config.json\').read_text())\n        super().__init__(*args,**kwargs)\n        self.add_callback(\'on_train_start\',self.b_start)\n        self.add_callback(\'on_train_epoch_start\',self.b_epoch)\n        self.add_callback(\'on_train_batch_start\',self.b_batch_start)\n        self.add_callback(\'on_train_batch_end\',self.b_batch)\n        self.add_callback(\'on_fit_epoch_end\',self.b_epoch_end)\n    def get_model(self,cfg=None,weights=None,verbose=True):\n        if self.b_config[\'arm\']==\'baseline\': return super().get_model(cfg,weights,verbose)\n        model=self.set_model_names_for_load(BDetectionModel(cfg,nc=self.data[\'nc\'],ch=self.data[\'channels\'],verbose=verbose))\n        if weights: model.load(weights)\n        return model\n    def b_start(self,t):\n        self.b_rows=[]; self.b_epoch_times=[]; self.b_context=None\n        if self.b_config[\'arm\']!=\'baseline\':\n            self.b_context=BContext(self,self.b_config[\'arm\'])\n            object.__setattr__(self.model,\'_b_context\',self.b_context)\n        self.b_ever_batch_size=self.batch_size\n        groups=[dict(lr=g[\'lr\'],initial_lr=g.get(\'initial_lr\'),weight_decay=g.get(\'weight_decay\'),betas=g.get(\'betas\'),momentum=g.get(\'momentum\'),parameters=len(g[\'params\'])) for g in self.optimizer.param_groups]\n        Path(\'results\').mkdir(exist_ok=True)\n        Path(\'results/optimizer-effective.json\').write_text(json.dumps(dict(type=type(self.optimizer).__name__,groups=groups,warmup_bias_lr=self.args.warmup_bias_lr),indent=2))\n        Path(\'results/resume-observed.json\').write_text(json.dumps(dict(start_epoch=self.start_epoch,optimizer_state_entries=len(self.optimizer.state),teacher_updates=self.ema.updates,initial_ramp=ramp(self.start_epoch,0,len(self.train_loader)))))\n    def b_epoch(self,t):\n        self._oom_retries=3  # refuse stock silent per-arm auto batch reduction\n        self.b_epoch_start=time.perf_counter()\n        self.b_batch_times=[]\n        if self.b_context: self.b_context.batch_index=0\n    def b_batch_start(self,t):\n        if self.b_config[\'smoke\'] and torch.cuda.is_available(): torch.cuda.synchronize()\n        self.b_batch_started=time.perf_counter()\n    def b_batch(self,t):\n        assert self.batch_size==self.b_ever_batch_size,\'Per-arm batch size changed\'\n        if self.b_config[\'smoke\'] and torch.cuda.is_available(): torch.cuda.synchronize()\n        self.b_batch_times.append(time.perf_counter()-self.b_batch_started)\n        if self.b_context:\n            row=self.b_context.last.copy(); row[\'teacher_updates_after\']=self.ema.updates\n            assert row[\'teacher_updates_after\']-row[\'teacher_updates_before\'] in [0,1]\n            row[\'supervised_items\']={k:float(v) for k,v in self.loss_items.items()}\n            row[\'total_batch_loss\']=float(self.loss.detach())\n            self.b_rows.append(row)\n            with open(\'results/aux-batches.jsonl\',\'a\') as f: f.write(json.dumps(row)+\'\\n\')\n    def b_epoch_end(self,t):\n        # Native final_eval emits a separate callback at epoch=epochs for best-checkpoint metrics.\n        if self.epoch>=self.epochs: return\n        rows=[r for r in self.b_rows if r[\'epoch\']==self.epoch+1]\n        record=dict(epoch=self.epoch+1,seconds=time.perf_counter()-self.b_epoch_start,\n                    allocated=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0,\n                    reserved=torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0,\n                    batches=len(rows),teacher_updates=self.ema.updates)\n        if len(self.b_batch_times)>2:\n            record[\'steady_batch_median_seconds\']=statistics.median(self.b_batch_times[2:])\n        if rows:\n            for key in [\'total\',\'visible\',\'target\',\'selected\',\'usable\',\'skip_no_positive\',\'skip_zero_weight\']:\n                record[key]=sum(r[key] for r in rows)\n            for key in [\'aux\',\'kl\',\'box_l1\',\'weight\']:\n                record[key+\'_mean\']=sum(r[key] for r in rows)/len(rows)\n            record[\'empty_selection_batches\']=sum(r[\'selected\']==0 for r in rows)\n            for key in [\'selected_by_class\',\'usable_by_class\']:\n                record[key]=[sum(r[key][c] for r in rows) for c in range(4)]\n        self.b_epoch_times.append(record)\n        Path(\'results/timing.json\').write_text(json.dumps(self.b_epoch_times,indent=2))\n')
Path('b_run.py').write_text('"""Kaggle stages for B, with an explicit benchmark/budget gate."""\nimport argparse\nimport copy\nimport gc\nimport gzip\nimport hashlib\nimport json\nimport os\nimport shutil\nimport statistics\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO,__version__\nfrom ultralytics.nn.tasks import DetectionModel\nfrom b_core import BTrainer,BDetectionModel,ramp\nimport pilot\nfrom metrics import operating,select_threshold,masks,xyxy,match\n\nROOT=Path(\'/kaggle/working\'); OUT=ROOT/\'results\'; OUT.mkdir(exist_ok=True)\nCFG=json.loads(Path(\'experiment.json\').read_text()); ARMS=CFG[\'arms\']\n\ndef save(name,data): (OUT/name).write_text(json.dumps(data,indent=2,default=str))\ndef sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef find(name):\n    files=list(Path(\'/kaggle/input\').rglob(name))\n    if not files: raise FileNotFoundError(name)\n    if len(files)>1:\n        assert len({sha(p) for p in files})==1,(\'Conflicting inputs\',name)\n    return files[0]\n\ndef setup():\n    assert __version__==CFG[\'ultralytics\'] and torch.cuda.is_available()\n    if not (ROOT/\'prepared/images\').exists():\n        with tarfile.open(find(\'prepared.tar.gz\')) as f: f.extractall(ROOT,filter=\'data\')\n    manifest=ROOT/\'prepared/manifest.json\'; assert sha(manifest)==CFG[\'manifest_sha256\']\n    rows=json.loads(manifest.read_text())\n    assert sum(r[\'split\']==\'train\' for r in rows)==8264\n    assert all(r[\'country\']!=\'Japan\' for r in rows)\n    save(\'environment.json\',dict(ultralytics=__version__,torch=torch.__version__,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory))\n    (OUT/\'pip-freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    return rows\n\ndef coco():\n    source=find(\'yolo11s.pt\'); assert sha(source)==CFG[\'coco_sha256\']\n    target=ROOT/\'yolo11s.pt\'; shutil.copy2(source,target)\n    return str(target)\n\ndef subset(rows):\n    train=[r for r in rows if r[\'split\']==\'train\']\n    def key(r): return hashlib.sha256(r[\'id\'].encode()).hexdigest()\n    target=[]\n    for r in train:\n        mm=masks(xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']),r[\'width\'],r[\'height\'])\n        if mm[\'small_or_thin\'].any(): target.append(r)\n    selected={r[\'id\']:r for r in sorted(target,key=key)[:128]}\n    for r in sorted(train,key=lambda r:(-len(r[\'boxes\']),key(r)))[:64]: selected[r[\'id\']]=r\n    for r in sorted(train,key=key):\n        if len(selected)>=256: break\n        selected[r[\'id\']]=r\n    ids=sorted(selected)\n    val=sorted([r for r in rows if r[\'split\']==\'selection\'],key=key)[:64]\n    (ROOT/\'prepared/b-smoke.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f\'{i}.jpg\') for i in ids))\n    (ROOT/\'prepared/b-smoke-val.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in val))\n    save(\'smoke-subset.json\',dict(train_ids=ids,selection_ids=[r[\'id\'] for r in val]))\n    return pilot.data_yaml(\'b-smoke\',\'b-smoke-val\',\'-b-smoke\')\n\ndef configuration(arm,batch,smoke=False):\n    d=pilot.settings(batch,2 if smoke else 50)\n    d.update(project=str(ROOT/\'training\'),name=arm,close_mosaic=0 if smoke else 10,\n             model=coco(),verbose=False)\n    if smoke: d[\'data\']=str(ROOT/\'data-b-smoke.yaml\')\n    Path(\'b-config.json\').write_text(json.dumps(dict(arm=arm,smoke=smoke,experiment=CFG)))\n    return d\n\ndef export_checkpoint(native,target):\n    """EMA weights become a normal stock DetectionModel; training-only subclass removed."""\n    source=YOLO(native).model.to(\'cuda\').float().eval()\n    plain=copy.deepcopy(source); plain.__class__=DetectionModel; plain.criterion=None\n    if hasattr(plain,\'_b_context\'): delattr(plain,\'_b_context\')\n    x=torch.zeros(1,3,640,640,device=\'cuda\')\n    with torch.no_grad(): a=source(x)[0]; b=plain(x)[0]\n    assert torch.allclose(a,b,atol=1e-6,rtol=1e-6),\'Export changed predictions\'\n    ckpt=torch.load(native,map_location=\'cpu\',weights_only=False)\n    checkpoint=dict(model=plain.cpu().half(),ema=None,optimizer=None,epoch=-1,train_args=ckpt[\'train_args\'],version=__version__,license=\'AGPL-3.0\')\n    torch.save(checkpoint,target)\n    restored=YOLO(target).model.to(\'cuda\').float().eval()\n    with torch.no_grad(): c=restored(x)[0]\n    assert torch.allclose(a,c,atol=1e-6,rtol=1e-6),\'Export roundtrip changed predictions\'\n    del source,plain,restored; gc.collect(); torch.cuda.empty_cache()\n    return sha(target)\n\nclass PlannedStop(Exception): pass\n\ndef run_smoke_arm(arm,batch):\n    torch.cuda.reset_peak_memory_stats()\n    args=configuration(arm,batch,True)\n    model=YOLO(args.pop(\'model\'))\n    def interrupt(trainer):\n        if trainer.epoch==0:\n            # on_model_save precedes on_fit_epoch_end; preserve first-epoch telemetry before stopping.\n            trainer.b_epoch_end(trainer)\n            raise PlannedStop(\'Intentional optimizer/EMA recovery test\')\n    model.add_callback(\'on_model_save\',interrupt)\n    before=time.perf_counter()\n    try: model.train(trainer=BTrainer,**args)\n    except PlannedStop: pass\n    else: raise AssertionError(\'Intentional stop did not fire\')\n    checkpoint=ROOT/\'training\'/arm/\'weights/last.pt\'\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    assert state[\'epoch\']==0 and state[\'optimizer\'] is not None and state[\'updates\']>0\n    first=json.loads((OUT/\'timing.json\').read_text())\n    del model; gc.collect(); torch.cuda.empty_cache()\n    model=YOLO(checkpoint); model.train(trainer=BTrainer,resume=True,device=0)\n    observed=json.loads((OUT/\'resume-observed.json\').read_text())\n    assert observed[\'start_epoch\']==1 and observed[\'optimizer_state_entries\']>0\n    assert observed[\'teacher_updates\']==state[\'updates\'] and observed[\'initial_ramp\']==.2\n    timings=first+json.loads((OUT/\'timing.json\').read_text())\n    unique={}\n    for t in timings: unique.setdefault(t[\'epoch\'],t)\n    assert set(unique)=={1,2}\n    digest=export_checkpoint(ROOT/\'training\'/arm/\'weights/best.pt\',OUT/f\'{arm}-smoke-export.pt\')\n    result=dict(arm=arm,batch=batch,seconds=time.perf_counter()-before,timing=list(unique.values()),resume=observed,export_sha256=digest)\n    if arm!=\'baseline\':\n        result[\'usable\']=sum(t.get(\'usable\',0) for t in unique.values())\n        result[\'selected\']=sum(t.get(\'selected\',0) for t in unique.values())\n        assert result[\'usable\']>0,(\'No usable consistency signal\',arm,result)\n    save(f\'smoke-{arm}.json\',result)\n    del model,state; gc.collect(); torch.cuda.empty_cache()\n    return result\n\ndef smoke(rows):\n    subprocess.run([sys.executable,\'-m\',\'unittest\',\'test_b_core\',\'-v\'],check=True)\n    subset(rows); attempts=[]\n    for batch in CFG[\'batch_candidates\']:\n        results=[]\n        try:\n            for arm in ARMS:\n                # Remove prior arm\'s temporary telemetry; keep dedicated smoke record.\n                for name in [\'aux-batches.jsonl\',\'timing.json\']:\n                    (OUT/name).unlink(missing_ok=True)\n                results.append(run_smoke_arm(arm,batch))\n        except RuntimeError as e:\n            if not any(s in str(e).lower() for s in [\'out of memory\',\'cuda backend memory\',\'cublas_status_alloc_failed\']): raise\n            attempts.append(dict(batch=batch,error=str(e)))\n            gc.collect(); torch.cuda.empty_cache(); continue\n        base=results[0][\'timing\'][-1][\'steady_batch_median_seconds\']\n        ratios={r[\'arm\']:r[\'timing\'][-1][\'steady_batch_median_seconds\']/base for r in results}\n        measured=sum(r[\'seconds\'] for r in results)/3600\n        projected=1.2*(1.52815234*sum(ratios.values()))+measured+1.0\n        gate=dict(status=\'passed\' if projected<=12 else \'budget_exceeded\',batch=batch,ratios=ratios,\n                  projected_gpu_hours=projected,budget=12,attempts=attempts,arms=results,\n                  benchmark_note=\'Median synchronized batch times after first2 batches of resumed smoke epoch; conservative20% contingency plus1h selection/evaluation. Baseline extrapolation1.528h. Verify full-run speed.\')\n        save(\'budget-gate.json\',gate); print(json.dumps(gate,indent=2),flush=True)\n        if projected>12: raise RuntimeError(\'Full comparison exceeds authorised12h estimate; revise budget before launching\')\n        return\n    raise RuntimeError(\'Consistency cannot fit at either permitted common batch\')\n\ndef train(rows,arm):\n    gate=json.loads(find(\'budget-gate.json\').read_text()); assert gate[\'status\']==\'passed\'\n    batch=gate[\'batch\']; args=configuration(arm,batch)\n    model=YOLO(args.pop(\'model\')); started=time.perf_counter()\n    model.train(trainer=BTrainer,**args)\n    assert model.trainer.batch_size==batch\n    native=ROOT/\'training\'/arm/\'weights/best.pt\'\n    digest=export_checkpoint(native,OUT/f\'{arm}-best.pt\')\n    shutil.copy2(ROOT/\'training\'/arm/\'results.csv\',OUT/f\'{arm}-history.csv\')\n    shutil.copy2(ROOT/\'training\'/arm/\'args.yaml\',OUT/f\'{arm}-args.yaml\')\n    # Preserve optimizer-bearing recovery checkpoints on Kaggle; exported best is for inference.\n    if (OUT/\'aux-batches.jsonl\').exists():\n        with open(OUT/\'aux-batches.jsonl\',\'rb\') as src,gzip.open(OUT/\'aux-batches.jsonl.gz\',\'wb\',compresslevel=3) as dst: shutil.copyfileobj(src,dst)\n        (OUT/\'aux-batches.jsonl\').unlink()\n    record=dict(status=\'complete\',arm=arm,epochs=50,batch=batch,seed=42,seconds=time.perf_counter()-started,\n                best_sha256=digest,coco_sha256=CFG[\'coco_sha256\'],manifest_sha256=CFG[\'manifest_sha256\'],experiment=CFG)\n    save(f\'{arm}-record.json\',record)\n\ndef select(rows):\n    selection=[r for r in rows if r[\'split\']==\'selection\']; assert len(selection)==1034\n    frozen={}\n    for arm in ARMS:\n        record=json.loads(find(f\'{arm}-record.json\').read_text()); assert record[\'status\']==\'complete\'\n        checkpoint=find(f\'{arm}-best.pt\'); assert sha(checkpoint)==record[\'best_sha256\']\n        model=YOLO(checkpoint)\n        # Compare the exported detector to the native selected EMA on actual source images.\n        # Native weights remain upstream for recovery; no audit/US images enter this check.\n        from PIL import Image\n        from ultralytics.data.augment import LetterBox\n        candidates=[p for p in Path(\'/kaggle/input\').rglob(\'best.pt\') if p.parent.parent.name==arm]\n        assert len(candidates)==1,(\'Missing/ambiguous native checkpoint\',arm,candidates)\n        native=YOLO(candidates[0]).model.to(\'cuda\').float().eval()\n        exported=model.model.to(\'cuda\').float().eval(); parity=[]\n        for r in sorted(selection,key=lambda r:r[\'id\'])[:3]:\n            pixels=np.asarray(Image.open(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg").convert(\'RGB\'))\n            pixels=LetterBox((640,640),auto=False)(image=pixels)\n            x=torch.from_numpy(np.ascontiguousarray(pixels.transpose(2,0,1))).to(\'cuda\').float()[None]/255\n            with torch.no_grad(): a=native(x)[0]; b=exported(x)[0]\n            assert torch.allclose(a,b,atol=1e-6,rtol=1e-6),(\'Source-image export mismatch\',arm,r[\'id\'])\n            parity.append(dict(id=r[\'id\'],max_absolute_difference=float((a-b).abs().max())))\n        save(f\'export-parity-{arm}.json\',parity)\n        del native; gc.collect(); torch.cuda.empty_cache()\n        predictions=pilot.predict(model,selection,640)\n        threshold,decision=select_threshold(selection,predictions)\n        frozen[arm]=dict(threshold=threshold,decision=decision,checkpoint_sha256=sha(checkpoint),record=record)\n        pilot.save(f\'predictions-selection-{arm}.json\',predictions)\n        del model; gc.collect(); torch.cuda.empty_cache()\n    save(\'frozen-models.json\',dict(arms=frozen,selection_ids=[r[\'id\'] for r in selection],manifest_sha256=CFG[\'manifest_sha256\']))\n\ndef evaluate(rows):\n    frozen=json.loads(find(\'frozen-models.json\').read_text()); assert set(frozen[\'arms\'])==set(ARMS)\n    assert set(frozen[\'selection_ids\'])=={r[\'id\'] for r in rows if r[\'split\']==\'selection\'}\n    summary={}; base_threshold=frozen[\'arms\'][\'baseline\'][\'threshold\']\n    for arm in ARMS:\n        spec=frozen[\'arms\'][arm]; checkpoint=find(f\'{arm}-best.pt\'); assert sha(checkpoint)==spec[\'checkpoint_sha256\']\n        model=YOLO(checkpoint); summary[arm]={}\n        for country in [\'India\',\'Czech\',\'United_States\']:\n            subset=[r for r in rows if r[\'country\']==country and r[\'split\'] in [\'source_audit\',\'unseen\']]\n            pred=pilot.predict(model,subset,640); pilot.save(f\'predictions-{arm}-{country}.json\',pred)\n            op=operating(subset,pred,spec[\'threshold\']); op[\'common_threshold\']=operating(subset,pred,base_threshold)\n            op[\'byclass\']={}\n            for cls in range(4):\n                tp=fp=gt_count=target_tp=target_count=0\n                for r in subset:\n                    gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(pred[r[\'id\']],float).reshape(-1,6)\n                    pairs,false,_=match(gt,pp,spec[\'threshold\']); target=masks(gt,r[\'width\'],r[\'height\'])[\'small_or_thin\']\n                    gt_count+=int((gt[:,0]==cls).sum()); tp+=sum(gt[j,0]==cls for _,j in pairs); fp+=sum(pp[i,5]==cls for i in false)\n                    target_count+=int(((gt[:,0]==cls)&target).sum()); target_tp+=sum(gt[j,0]==cls and target[j] for _,j in pairs)\n                op[\'byclass\'][cls]=dict(gt=int(gt_count),tp=int(tp),fp=int(fp),target_gt=int(target_count),target_tp=int(target_tp))\n            # Score all labels with exactly the previous baseline\'s evaluator configuration.\n            (ROOT/\'prepared\'/f\'{country}.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in subset))\n            val=model.val(data=pilot.data_yaml(val=country,suffix=\'-\'+country),imgsz=640,rect=False,batch=8,device=0,conf=.001,iou=.7,max_det=300,plots=False,verbose=False,project=str(ROOT/\'validation\'),name=f\'{arm}-{country}\')\n            op[\'AP\']=dict(map50=float(val.box.map50),map50_95=float(val.box.map),per_class=val.box.all_ap.tolist(),class_index=val.box.ap_class_index.tolist())\n            # Hash-selected examples, aligned across arms and labelled by image ID.\n            from PIL import Image,ImageDraw\n            for r in sorted(subset,key=lambda r:hashlib.sha256(r[\'id\'].encode()).hexdigest())[:6]:\n                im=Image.open(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg").convert(\'RGB\'); draw=ImageDraw.Draw(im)\n                gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(pred[r[\'id\']],float).reshape(-1,6)\n                _,_,miss=match(gt,pp,spec[\'threshold\'])\n                for j,b in enumerate(gt): draw.rectangle(b[1:].tolist(),outline=\'red\' if j in miss else \'green\',width=3)\n                for b in pp:\n                    if b[4]>=spec[\'threshold\']: draw.rectangle(b[:4].tolist(),outline=\'cyan\',width=2)\n                im.thumbnail((960,960)); im.save(OUT/f\'example-{arm}-{r["id"]}.jpg\')\n            summary[arm][country]=op; save(\'b-summary.json\',summary)\n        del model; gc.collect(); torch.cuda.empty_cache()\n    save(\'evaluation-complete.json\',dict(status=\'complete\',frozen=frozen,experiment=CFG))\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(); parser.add_argument(\'stage\',choices=[\'smoke\',\'train\',\'select\',\'evaluate\']); parser.add_argument(\'--arm\',choices=ARMS)\n    args=parser.parse_args(); stage_started=time.perf_counter(); rows=setup()\n    try:\n        if args.stage==\'train\': train(rows,args.arm)\n        else: globals()[args.stage](rows)\n    finally:\n        save(\'stage-resources.json\',dict(stage=args.stage,arm=args.arm,seconds_including_preparation=time.perf_counter()-stage_started,\n             allocated=torch.cuda.max_memory_allocated(),reserved=torch.cuda.max_memory_reserved(),\n             working_bytes_before_cleanup=sum(p.stat().st_size for p in ROOT.rglob(\'*\') if p.is_file())))\n        shutil.rmtree(ROOT/\'prepared\',ignore_errors=True)\n')
Path('test_b_core.py').write_text("import copy\nimport unittest\nimport torch\nfrom ultralytics.cfg import get_cfg\nfrom ultralytics.nn.tasks import DetectionModel\nfrom b_core import agreement,geometry_selection,top_three,ramp,BDetectionModel,consistency,BTrainer\n\ntorch.set_num_threads(2)\n\nclass CoreTests(unittest.TestCase):\n    def test_identical_zero_and_independent_sigmoids(self):\n        p=torch.tensor([[[.9,.8,.1,.3]]])  # does not sum to one\n        b=torch.tensor([[[.1,.2,.3,.4]]])\n        loss,_,_=agreement(p,b,p.clone(),b.clone())\n        self.assertTrue(torch.allclose(loss,torch.zeros_like(loss),atol=1e-7))\n    def test_teacher_detached_student_has_gradients(self):\n        p=torch.tensor([[[.8,.7,.2,.1]]],requires_grad=True)\n        q=torch.tensor([[[.3,.4,.5,.6]]],requires_grad=True)\n        b=torch.tensor([[[.1,.2,.3,.4]]],requires_grad=True)\n        s=torch.tensor([[[.2,.1,.4,.5]]],requires_grad=True)\n        agreement(p,b,q,s)[0].sum().backward()\n        self.assertIsNone(p.grad); self.assertIsNone(b.grad)\n        self.assertGreater(float(q.grad.abs().sum()),0); self.assertGreater(float(s.grad.abs().sum()),0)\n    def test_box_normalization(self):\n        full=torch.tensor([[[64.,128.,256.,320.]]]); half=full/2\n        p=torch.full((1,1,4),.5)\n        self.assertEqual(float(agreement(p,full/640,p,half/320)[0]),0.)\n    def test_geometry_boundaries_and_union(self):\n        batch={'batch_idx':torch.tensor([0,0,0,0]),'bboxes':torch.tensor([[.5,.5,16/640,64/640],[.5,.5,32/640,32/640],[.5,.5,8/640,10/640],[.5,.5,7/640,10/640]])}\n        selected,stats=geometry_selection(batch,1,4,True)\n        self.assertEqual(selected.tolist(),[[False,False,True,False]])\n        self.assertEqual(stats['target'],1)\n        generic,_=geometry_selection(batch,1,4,False)\n        self.assertEqual(generic.tolist(),[[True,True,True,False]])\n    def test_top_three_ties_and_fewer_candidates(self):\n        view=dict(probabilities=torch.tensor([[[.1],[.2],[.3],[.9]]],requires_grad=True),boxes=torch.zeros(1,4,4),\n                  fg=torch.ones(1,4,dtype=torch.bool),indices=torch.zeros(1,4,dtype=torch.long),alignment=torch.ones(1,1,4))\n        p,b,valid,positive=top_three(view)\n        self.assertAlmostEqual(float(p.detach()),.2,places=6)\n        p.sum().backward(); self.assertEqual(float(view['probabilities'].grad[0,3,0]),0.)\n        view['fg'][:]=False\n        p,b,valid,positive=top_three(view)\n        self.assertFalse(bool(valid.any())); self.assertFalse(bool(positive.any()))\n        self.assertTrue(torch.isfinite(p).all())\n    def test_ramp_resume(self):\n        self.assertEqual(ramp(0,0,10),0); self.assertEqual(ramp(1,0,10),.2)\n        self.assertEqual(ramp(5,0,10),1)\n    def test_final_validation_is_not_an_extra_training_epoch(self):\n        trainer=object.__new__(BTrainer); trainer.epoch=2; trainer.epochs=2; trainer.b_epoch_times=[]\n        trainer.b_epoch_end(trainer)\n        self.assertEqual(trainer.b_epoch_times,[])\n    def test_stock_loss_and_grad_parity_without_context(self):\n        torch.manual_seed(42)\n        stock=DetectionModel('yolo11s.yaml',nc=4,verbose=False)\n        stock.args=get_cfg(overrides=dict(task='detect',imgsz=640))\n        wrapped=copy.deepcopy(stock); wrapped.__class__=BDetectionModel\n        stock.train(); wrapped.train()\n        batch=dict(img=torch.rand(2,3,64,64),batch_idx=torch.tensor([0.,1.]),cls=torch.tensor([[0.],[3.]]),bboxes=torch.tensor([[.5,.5,.3,.3],[.4,.4,.2,.2]]))\n        a,ai=stock.loss(batch); b,bi=wrapped.loss(batch)\n        self.assertTrue(torch.equal(a,b))\n        a.sum().backward(); b.sum().backward()\n        for p,q in zip(stock.parameters(),wrapped.parameters()):\n            if p.grad is not None: self.assertTrue(torch.equal(p.grad,q.grad))\n    def test_tal_decode_consistency_gradients_and_empty_batch(self):\n        model=DetectionModel('yolo11s.yaml',nc=4,verbose=False); model.args=get_cfg()\n        criterion=model.init_criterion()\n        def raw(size,teacher):\n            shapes=[size//8,size//16,size//32]; n=sum(s*s for s in shapes)\n            d=torch.full((1,4,16,n),-15.)\n            if teacher: d[:,:,1,:]=15.\n            else: d[:,:,:2,:]=0.\n            scores=torch.full((1,4,n),-.8 if teacher else -.4); scores[:,0]=1.4 if teacher else .4\n            return dict(boxes=d.reshape(1,64,n).requires_grad_(),scores=scores.requires_grad_(),feats=[torch.zeros(1,1,s,s) for s in shapes])\n        batch=dict(img=torch.zeros(1,3,640,640),batch_idx=torch.tensor([0.]),cls=torch.tensor([[0.]]),bboxes=torch.tensor([[100/640,100/640,16/640,16/640]]))\n        student=raw(320,False); teacher=raw(640,True)\n        loss,stats=consistency(student,teacher,batch,criterion,True)\n        self.assertEqual(stats['usable'],1); self.assertGreater(float(loss.detach()),0)\n        loss.backward()\n        self.assertIsNone(teacher['scores'].grad); self.assertIsNone(teacher['boxes'].grad)\n        self.assertGreater(float(student['scores'].grad.abs().sum()),0)\n        self.assertGreater(float(student['boxes'].grad.abs().sum()),0)\n        batch.update(batch_idx=torch.zeros(0),cls=torch.zeros(0,1),bboxes=torch.zeros(0,4))\n        loss,stats=consistency(raw(320,False),raw(640,True),batch,criterion,True)\n        self.assertEqual(stats['usable'],0); self.assertEqual(float(loss.detach()),0)\n        loss.backward()\n    def test_auxiliary_is_added_once_not_three_times(self):\n        model=BDetectionModel('yolo11s.yaml',nc=4,verbose=False); model.args=get_cfg()\n        model.train()\n        batch=dict(img=torch.rand(2,3,64,64),batch_idx=torch.tensor([0.,1.]),cls=torch.tensor([[0.],[3.]]),bboxes=torch.tensor([[.5,.5,.3,.3],[.4,.4,.2,.2]]))\n        base,_=model.loss(batch)\n        class FakeContext:\n            def compute(self,m,b): return b['img'].sum()*0+2\n        object.__setattr__(model,'_b_context',FakeContext())\n        extended,_=model.loss(batch)\n        self.assertAlmostEqual(float((extended.sum()-base.sum()).detach()),4.,places=5)\n\nif __name__=='__main__': unittest.main()\n")
Path('experiment.json').write_text('{\n  "arms": ["baseline", "generic", "small_thin"],\n  "seed": 42,\n  "epochs": 50,\n  "imgsz": 640,\n  "half_size": 320,\n  "batch_candidates": [16, 8],\n  "maximum_lambda": 1.0,\n  "ramp_epochs": 5,\n  "top_k": 3,\n  "budget_gpu_hours": 12,\n  "manifest_sha256": "3e4eeb11397ec5cacbf7533b558bb80d50eb5f16dd670ce85dd5d19be1f47123",\n  "coco_sha256": "85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5",\n  "ultralytics": "8.4.192",\n  "source_train_images": 8264,\n  "source_selection_images": 1034,\n  "source_audit_images": 1033,\n  "us_development_images": 4805\n}\n')
Path('metrics.py').write_text('"""Explicit operational matching/geometry; AP is computed by the pinned YOLO validator."""\nimport numpy as np\n\ndef grid_cell(x,y,width,height):\n    gain=640/max(width,height)\n    # Ultralytics LetterBox rounds resized dimensions then splits integer padding.\n    left=round((640-round(width*gain))/2-.1)\n    top=round((640-round(height*gain))/2-.1)\n    return min(39,max(0,int((y*gain+top)/16))),min(39,max(0,int((x*gain+left)/16)))\n\ndef xyxy(boxes, width, height):\n    out=[]\n    for c,x,y,w,h in boxes:\n        out.append([c,(x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height])\n    return np.asarray(out,dtype=float).reshape(-1,5)\n\ndef iou(a,b):\n    lo=np.maximum(a[:2],b[:,:2]); hi=np.minimum(a[2:],b[:,2:])\n    inter=np.maximum(0,hi-lo).prod(1)\n    area=np.maximum(0,a[2:]-a[:2]).prod()\n    return inter/(area+np.maximum(0,b[:,2:]-b[:,:2]).prod(1)-inter+1e-12)\n\ndef match(gt,pred,threshold,overlap=.5):\n    # GT: class,x1,y1,x2,y2; prediction: x1,y1,x2,y2,confidence,class.\n    used=set(); pairs=[]; fp=[]\n    for i in sorted(range(len(pred)),key=lambda i:-pred[i,4]):\n        if pred[i,4]<threshold: continue\n        eligible=[j for j in range(len(gt)) if j not in used and gt[j,0]==pred[i,5]]\n        if eligible:\n            overlaps=iou(pred[i,:4],gt[eligible,1:]); j=int(np.argmax(overlaps))\n            if overlaps[j]>=overlap:\n                used.add(eligible[j]); pairs.append((i,eligible[j])); continue\n        fp.append(i)\n    return pairs,fp,[j for j in range(len(gt)) if j not in used]\n\ndef masks(gt,width,height):\n    # Canonicalise sub-micropixel cancellation from xywh -> xyxy -> side lengths.\n    # Exact 16px/32px boundaries must not drift below strict cutoffs.\n    sides=np.round((gt[:,3:5]-gt[:,1:3])*640/max(width,height),6)\n    small=sides.prod(1)<32**2; thin=sides.min(1)<16\n    return dict(all=np.ones(len(gt),dtype=bool),small=small,thin=thin,small_or_thin=small|thin,\n                B_eligible=(small|thin)&(sides.min(1)>=8))\n\ndef operating(records,predictions,threshold):\n    totals={k:dict(tp=0,gt=0) for k in [\'all\',\'small\',\'thin\',\'small_or_thin\',\'B_eligible\']}\n    fps=0; damage_images=0; images_with_miss=0; per_image=[]\n    for r in records:\n        gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pred=np.asarray(predictions[r[\'id\']],float).reshape(-1,6)\n        pairs,fp,miss=match(gt,pred,threshold); mm=masks(gt,r[\'width\'],r[\'height\']); hit={j for _,j in pairs}\n        for k,m in mm.items():\n            totals[k][\'gt\']+=int(m.sum()); totals[k][\'tp\']+=sum(bool(m[j]) for j in hit)\n        fps+=len(fp); damage_images+=bool(len(gt)); images_with_miss+=bool(miss)\n        per_image.append(dict(id=r[\'id\'],group=r[\'group\'],tp=len(pairs),fp=len(fp),fn=len(miss),\n                              target_gt=int(mm[\'small_or_thin\'].sum()),target_tp=sum(bool(mm[\'small_or_thin\'][j]) for j in hit)))\n    for v in totals.values(): v[\'recall\']=v[\'tp\']/v[\'gt\'] if v[\'gt\'] else None; v[\'sparse\']=v[\'gt\']<30\n    tp=totals[\'all\'][\'tp\']; fn=totals[\'all\'][\'gt\']-tp\n    target_fn=totals[\'small_or_thin\'][\'gt\']-totals[\'small_or_thin\'][\'tp\']\n    return dict(threshold=threshold,geometry=totals,precision=tp/(tp+fps) if tp+fps else 0,\n                fp_per_image=fps/max(1,len(records)),misses_per_image=fn/max(1,len(records)),\n                small_or_thin_share_of_misses=target_fn/fn if fn else None,\n                small_or_thin_share_of_labels=totals[\'small_or_thin\'][\'gt\']/totals[\'all\'][\'gt\'] if totals[\'all\'][\'gt\'] else None,\n                damage_images_with_miss_fraction=images_with_miss/damage_images if damage_images else None,\n                images=len(records),per_image=per_image)\n\ndef select_threshold(records,predictions):\n    # Fixed reproducible grid; predictions below .001 are not retained.\n    rows=[]\n    for t in np.linspace(.001,.999,200):\n        m=operating(records,predictions,float(t)); p=m[\'precision\']; rec=m[\'geometry\'][\'all\'][\'recall\'] or 0\n        rows.append((float(t),p,rec,2*p*rec/(p+rec) if p+rec else 0))\n    eligible=[r for r in rows if r[1]>=.7]\n    chosen=max(eligible,key=lambda r:(r[2],r[0])) if eligible else max(rows,key=lambda r:(r[3],r[0]))\n    return chosen[0],dict(rule=\'precision>=0.70, maximum recall, ties higher threshold\' if eligible else \'best F1 fallback\',grid=rows)\n\ndef average_precision_binary(labels,scores):\n    labels=np.asarray(labels,bool); scores=np.asarray(scores,float)\n    if not labels.any(): return None\n    # Tie-aware stepwise average precision (same definition as sklearn average_precision_score).\n    order=np.argsort(-scores,kind=\'stable\'); y=labels[order]; s=scores[order]\n    ends=np.r_[np.where(np.diff(s)!=0)[0],len(s)-1]\n    tp=np.cumsum(y)[ends]; recall=tp/y.sum(); precision=tp/(ends+1)\n    return float(np.sum(np.diff(np.r_[0,recall])*precision))\n\ndef bootstrap(records,per_image,n=1000):\n    groups={}\n    for x in per_image: groups.setdefault(x[\'group\'],[]).append(x)\n    values=list(groups.values()); rng=np.random.default_rng(42); rows=[]\n    for _ in range(n):\n        sample=[x for i in rng.integers(0,len(values),len(values)) for x in values[i]]\n        tp=sum(x[\'tp\'] for x in sample); fp=sum(x[\'fp\'] for x in sample); fn=sum(x[\'fn\'] for x in sample)\n        tg=sum(x[\'target_gt\'] for x in sample); tt=sum(x[\'target_tp\'] for x in sample)\n        rows.append([tp/max(1,tp+fn),tp/max(1,tp+fp),fp/len(sample),tt/tg if tg else np.nan])\n    return {k:np.nanpercentile(np.asarray(rows)[:,i],[2.5,97.5]).tolist() for i,k in enumerate([\'recall\',\'precision\',\'fp_per_image\',\'small_or_thin_recall\'])}\n')
Path('pilot.py').write_text('"""Kaggle smoke/recovery, stock training and source-only evaluation."""\nimport argparse\nimport collections\nimport hashlib\nimport gzip\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport yaml\nfrom PIL import Image, ImageDraw\nfrom ultralytics import YOLO, __version__\nfrom metrics import xyxy,match,masks,operating,select_threshold,average_precision_binary,bootstrap,grid_cell\n\nROOT=Path(\'/kaggle/working\'); OUT=ROOT/\'results\'; OUT.mkdir(exist_ok=True)\n\ndef save(name,obj):\n    body=json.dumps(obj,indent=2,default=str)\n    if name.startswith(\'predictions-\'):\n        # Low-threshold full-country predictions can be large; compress transfer artifacts.\n        with gzip.open(OUT/(name+\'.gz\'),\'wt\',encoding=\'utf-8\',compresslevel=3) as f: f.write(body)\n    else: (OUT/name).write_text(body)\n\ndef setup():\n    assert __version__==\'8.4.192\', __version__\n    assert torch.cuda.is_available(), \'GPU required for this stage\'\n    if not (ROOT/\'prepared/images\').exists():\n        archive=next(Path(\'/kaggle/input\').rglob(\'prepared.tar.gz\'))\n        with tarfile.open(archive) as f: f.extractall(ROOT,filter=\'data\')\n    env=dict(ultralytics=__version__,python=platform.python_version(),torch=torch.__version__,\n             cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)\n    save(\'environment.json\',env)\n    (OUT/\'pip-freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    return json.loads((ROOT/\'prepared/manifest.json\').read_text())\n\ndef data_yaml(train=\'train\',val=\'selection\',suffix=\'\'):\n    p=ROOT/f\'data{suffix}.yaml\'\n    p.write_text(yaml.safe_dump(dict(path=str(ROOT/\'prepared\'),train=str(ROOT/\'prepared\'/f\'{train}.txt\'),\n                                   val=str(ROOT/\'prepared\'/f\'{val}.txt\'),names=[\'D00\',\'D10\',\'D20\',\'D40\'])))\n    return str(p)\n\ndef settings(batch,epochs=50):\n    return dict(data=data_yaml(),epochs=epochs,imgsz=640,batch=batch,device=0,amp=True,seed=42,\n                deterministic=True,workers=2,cache=False,save=True,save_period=5,patience=100,\n                project=str(ROOT/\'training\'),exist_ok=True,plots=True)\n\ndef smoke(records):\n    ids=[r[\'id\'] for r in records if r[\'split\']==\'train\']\n    # Deterministic small execution test, not the scientific pilot dataset.\n    ids=sorted(ids,key=lambda s:hashlib.sha256(s.encode()).hexdigest())[:128]\n    (ROOT/\'prepared/smoke.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f\'{i}.jpg\') for i in ids))\n    vals=[r[\'id\'] for r in records if r[\'split\']==\'selection\'][:64]\n    (ROOT/\'prepared/smokeval.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f\'{i}.jpg\') for i in vals))\n    smoke_data=data_yaml(\'smoke\',\'smokeval\',\'-smoke\')\n    class PlannedInterruption(Exception): pass\n    def interrupt(trainer):\n        if trainer.epoch==0: raise PlannedInterruption(\'Deliberate interruption after epoch checkpoint\')\n    attempts=[]\n    for modelname in [\'yolo11s.pt\',\'yolo11n.pt\']:\n        for batch in [16,8,4]:\n            model=YOLO(modelname); model.add_callback(\'on_model_save\',interrupt)\n            args=settings(batch,2); args.update(data=smoke_data,name=\'smoke\')\n            try:\n                model.train(**args)\n                raise RuntimeError(\'interruption callback did not run\')\n            except PlannedInterruption: pass\n            except torch.cuda.OutOfMemoryError:\n                attempts.append(dict(model=modelname,batch=batch,status=\'OOM\')); del model; torch.cuda.empty_cache(); continue\n            checkpoint=ROOT/\'training/smoke/weights/last.pt\'\n            state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n            assert state[\'epoch\']==0 and state[\'optimizer\'] is not None\n            resume_model=YOLO(checkpoint)\n            seen=[]\n            resume_model.add_callback(\'on_train_start\',lambda t:seen.append(dict(start_epoch=t.start_epoch,optimizer_state_entries=len(t.optimizer.state))))\n            resume_model.train(resume=True,device=0)\n            assert seen[0][\'start_epoch\']==1 and seen[0][\'optimizer_state_entries\']>0,seen\n            save(\'smoke.json\',dict(status=\'passed\',model=modelname,batch=batch,recovery=seen,attempts=attempts))\n            shutil.copy2(ROOT/\'training/smoke/results.csv\',OUT/\'smoke-training.csv\')\n            return\n    raise RuntimeError(\'Both model sizes failed at all batches\')\n\ndef assignment_diagnostics(model,records):\n    from ultralytics.utils.tal import TaskAlignedAssigner\n    original=TaskAlignedAssigner.forward; rows=[]\n    def wrapped(self,pd_scores,pd_bboxes,anc_points,gt_labels,gt_bboxes,mask_gt,**extra):\n        original_gt=gt_bboxes.detach().clone()\n        result=original(self,pd_scores,pd_bboxes,anc_points,gt_labels,gt_bboxes,mask_gt,**extra)\n        _,_,scores,fg,idx=result\n        for b in range(len(gt_labels)):\n            for j in range(gt_labels.shape[1]):\n                if not mask_gt[b,j].item(): continue\n                sel=fg[b]&(idx[b]==j); size=(original_gt[b,j,2:]-original_gt[b,j,:2]).detach().cpu().tolist()\n                rows.append(dict(cls=int(gt_labels[b,j].item()),width=size[0],height=size[1],positives=int(sel.sum()),score_mass=float(scores[b,sel].sum())))\n        return result\n    TaskAlignedAssigner.forward=wrapped\n    net=model.model.to(\'cuda\'); net.eval()\n    try:\n        from ultralytics.data import build_yolo_dataset,build_dataloader\n        args=model.overrides.copy()\n        from ultralytics.cfg import get_cfg\n        if isinstance(net.args,dict): net.args=get_cfg(overrides=net.args)\n        cfg=get_cfg(overrides=dict(imgsz=640,task=\'detect\',mode=\'train\',augment=False))\n        train=[]\n        for country in [\'India\',\'Czech\']:\n            train.extend(sorted([r for r in records if r[\'split\']==\'train\' and r[\'country\']==country],key=lambda r:hashlib.sha256(r[\'id\'].encode()).hexdigest())[:128])\n        txt=ROOT/\'assignment.txt\'; txt.write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in train))\n        ds=build_yolo_dataset(cfg,str(txt),8,dict(names={i:c for i,c in enumerate([\'D00\',\'D10\',\'D20\',\'D40\'])},nc=4,channels=3),mode=\'val\',rect=False,stride=32)\n        loader=build_dataloader(ds,8,0,shuffle=False)\n        with torch.no_grad():\n            for batch in loader:\n                batch={k:v.to(\'cuda\') if isinstance(v,torch.Tensor) else v for k,v in batch.items()}\n                batch[\'img\']=batch[\'img\'].float()/255\n                net.loss(batch)\n        summaries={}\n        for name,subset in [(\'all\',rows),(\'small_or_thin\',[r for r in rows if r[\'width\']*r[\'height\']<1024 or min(r[\'width\'],r[\'height\'])<16]),(\'other\',[r for r in rows if r[\'width\']*r[\'height\']>=1024 and min(r[\'width\'],r[\'height\'])>=16])]:\n            summaries[name]=dict(boxes=len(subset),zero_positive_fraction=sum(r[\'positives\']==0 for r in subset)/len(subset) if subset else None,\n                                  mean_positives=float(np.mean([r[\'positives\'] for r in subset])) if subset else None,\n                                  mean_score_mass=float(np.mean([r[\'score_mass\'] for r in subset])) if subset else None)\n        save(\'A-assignment.json\',dict(records=rows,summary=summaries,images=len(train),note=\'Post-training frozen-best checkpoint, float32 evaluation mode, no gradients or optimizer steps; training-sample supervision diagnostic, not a log of actual training-time assignments.\'))\n    finally: TaskAlignedAssigner.forward=original\n\ndef train(records):\n    smoke_file=next(Path(\'/kaggle/input\').rglob(\'smoke.json\'))\n    smoke_info=json.loads(smoke_file.read_text()); assert smoke_info[\'status\']==\'passed\'\n    model=YOLO(smoke_info[\'model\']); times=[]; started=time.time(); epoch_start=[0]\n    model.add_callback(\'on_train_epoch_start\',lambda t:epoch_start.__setitem__(0,time.time()))\n    def end(t):\n        times.append(dict(epoch=t.epoch+1,seconds=time.time()-epoch_start[0],allocated=torch.cuda.max_memory_allocated(),reserved=torch.cuda.max_memory_reserved()))\n        save(\'timing.json\',dict(epochs=times,elapsed_seconds=time.time()-started))\n    model.add_callback(\'on_fit_epoch_end\',end)\n    model.train(**settings(smoke_info[\'batch\']),name=\'baseline\')\n    save(\'training-complete.json\',dict(status=\'complete\',epochs=50,model=smoke_info[\'model\'],batch=smoke_info[\'batch\'],seconds=time.time()-started))\n    for name in [\'best.pt\',\'last.pt\']: shutil.copy2(ROOT/\'training/baseline/weights\'/name,OUT/name)\n    for name in [\'results.csv\',\'args.yaml\']: shutil.copy2(ROOT/\'training/baseline\'/name,OUT/name)\n    save(\'checkpoint-sha256.json\',{n:hashlib.sha256((OUT/n).read_bytes()).hexdigest() for n in [\'best.pt\',\'last.pt\']})\n\ndef predict(model,records,size):\n    data={}; paths=[str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in records]\n    # A Python list is interpreted as in-memory PIL/numpy input by Ultralytics,\n    # which can eagerly load all images and use one huge batch. A text source is lazy.\n    input_list=ROOT/\'prediction-inputs.txt\'; input_list.write_text(\'\\n\'.join(paths))\n    for result in model.predict(source=str(input_list),imgsz=size,rect=False,conf=.001,iou=.7,max_det=300,batch=8,device=0,stream=True,verbose=False):\n        boxes=result.boxes\n        data[Path(result.path).stem]=torch.cat([boxes.xyxy,boxes.conf[:,None],boxes.cls[:,None]],1).cpu().tolist()\n    return data\n\ndef c_diagnostics(records,preds,threshold):\n    labels=[]; scores=[]; positives=0; collisions=0; positive_images=0\n    for r in records:\n        gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pred=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n        _,_,miss=match(gt,pred,threshold); m=masks(gt,r[\'width\'],r[\'height\'])[\'small_or_thin\']\n        target=np.zeros((40,40),bool); weak=np.zeros((40,40))\n        # Grid is on square letterboxed 640 view, not a stretched original image.\n        def cell(x,y): return grid_cell(x,y,r[\'width\'],r[\'height\'])\n        for j in miss:\n            if not m[j]: continue\n            y,x=cell((gt[j,1]+gt[j,3])/2,(gt[j,2]+gt[j,4])/2)\n            collisions+=int(target[y,x]); target[y,x]=True\n        for p in pred:\n            if .05<=p[4]<threshold:\n                y,x=cell((p[0]+p[2])/2,(p[1]+p[3])/2); weak[y,x]=max(weak[y,x],p[4])\n        positives+=int(target.sum()); positive_images+=int(target.any()); labels.extend(target.ravel()); scores.extend(weak.ravel())\n    prevalence=positives/max(1,len(labels)); ap=average_precision_binary(labels,scores)\n    return dict(positive_cells=positives,positive_images=positive_images,collisions=collisions,prevalence=prevalence,\n                weak_map_average_precision=ap,ap_over_prevalence=ap/prevalence if ap is not None and prevalence else None,\n                weak_map_empty_by_threshold=threshold<=.05,grid=\'40x40 square-letterbox coordinate system\')\n\ndef evaluate(records):\n    checkpoint=next(p for p in Path(\'/kaggle/input\').rglob(\'best.pt\') if p.parent.name==\'results\')\n    coco=list(Path(\'/kaggle/input\').rglob(\'yolo11s.pt\'))\n    save(\'initial-weight-provenance.json\',dict(files={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in coco},\n         observed_training_download=\'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt\'))\n    model=YOLO(checkpoint); selected=[r for r in records if r[\'split\']==\'selection\']\n    # Keep diagnostics in the evaluation stage so library/API issues cannot spoil a completed training job.\n    assignment_diagnostics(YOLO(checkpoint),records)\n    assert all(r[\'country\'] in [\'India\',\'Czech\'] for r in selected)\n    selection_preds=predict(model,selected,640)\n    save(\'predictions-source-selection-640.json\',selection_preds)\n    threshold,decision=select_threshold(selected,selection_preds); save(\'threshold.json\',dict(value=threshold,selection_ids=[r[\'id\'] for r in selected],**decision))\n    summary={}; samples=[]\n    for country in [\'India\',\'Czech\',\'United_States\']:\n        subset=[r for r in records if r[\'country\']==country and r[\'split\'] in [\'source_audit\',\'unseen\']]\n        preds=predict(model,subset,640); half=predict(model,subset,320)\n        save(f\'predictions-{country}-640.json\',preds); save(f\'predictions-{country}-320.json\',half)\n        op=operating(subset,preds,threshold); op[\'bootstrap95\']=bootstrap(subset,op[\'per_image\']); op[\'C\']=c_diagnostics(subset,preds,threshold)\n        positive=[r for r in subset if r[\'boxes\']]; negative=[r for r in subset if not r[\'boxes\']]\n        op[\'positive_images_operating\']=operating(positive,preds,threshold) if positive else None\n        op[\'negative_images_operating\']=operating(negative,preds,threshold) if negative else None\n        op[\'scale320\']=operating(subset,half,threshold)\n        transitions={k:collections.Counter() for k in [\'all\',\'small_or_thin\',\'B_eligible\']}\n        for r in subset:\n            gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); mm=masks(gt,r[\'width\'],r[\'height\'])\n            hits=[]\n            for predictions in [preds,half]:\n                pp=np.asarray(predictions[r[\'id\']],float).reshape(-1,6)\n                hits.append({j for _,j in match(gt,pp,threshold)[0]})\n            for k,counts in transitions.items():\n                for j in np.flatnonzero(mm[k]):\n                    state=\'both\' if j in hits[0] and j in hits[1] else \'640_only\' if j in hits[0] else \'320_only\' if j in hits[1] else \'neither\'\n                    counts[state]+=1\n        op[\'B_scale_transitions\']={k:dict(v) for k,v in transitions.items()}\n        byclass={}\n        for cls in range(4):\n            counts=dict(gt=0,tp=0,fp=0,small_or_thin_gt=0,small_or_thin_tp=0)\n            for r in subset:\n                gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n                pairs,fp,_=match(gt,pp,threshold)\n                counts[\'gt\']+=int((gt[:,0]==cls).sum()); counts[\'tp\']+=sum(gt[j,0]==cls for _,j in pairs); counts[\'fp\']+=sum(pp[i,5]==cls for i in fp)\n                gm=masks(gt,r[\'width\'],r[\'height\'])[\'small_or_thin\']\n                counts[\'small_or_thin_gt\']+=int(((gt[:,0]==cls)&gm).sum())\n                counts[\'small_or_thin_tp\']+=sum(gt[j,0]==cls and gm[j] for _,j in pairs)\n            byclass[cls]={k:int(v) for k,v in counts.items()}\n            byclass[cls].update(precision=float(counts[\'tp\']/(counts[\'tp\']+counts[\'fp\'])) if counts[\'tp\']+counts[\'fp\'] else None,\n                                recall=float(counts[\'tp\']/counts[\'gt\']) if counts[\'gt\'] else None,sparse=counts[\'gt\']<30,\n                                small_or_thin_recall=float(counts[\'small_or_thin_tp\']/counts[\'small_or_thin_gt\']) if counts[\'small_or_thin_gt\'] else None,\n                                small_or_thin_sparse=counts[\'small_or_thin_gt\']<30)\n        op[\'byclass\']=byclass\n        (ROOT/\'prepared\'/f\'{country}.txt\').write_text(\'\\n\'.join(str(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg") for r in subset))\n        val=model.val(data=data_yaml(val=country,suffix=\'-\'+country),imgsz=640,rect=False,batch=8,device=0,conf=.001,iou=.7,max_det=300,plots=False,verbose=False,project=str(ROOT/\'validation\'),name=country)\n        op[\'AP\']=dict(map50=float(val.box.map50),map50_95=float(val.box.map),per_class_ap=val.box.all_ap.tolist(),ap_class_index=val.box.ap_class_index.tolist())\n        summary[country]=op\n        # Fixed hash sample: no cherry-picking by apparent severity.\n        for r in sorted(subset,key=lambda r:hashlib.sha256(r[\'id\'].encode()).hexdigest())[:12]:\n            im=Image.open(ROOT/\'prepared/images\'/f"{r[\'id\']}.jpg").convert(\'RGB\'); draw=ImageDraw.Draw(im)\n            gt=xyxy(r[\'boxes\'],r[\'width\'],r[\'height\']); pp=np.asarray(preds[r[\'id\']],float).reshape(-1,6)\n            _,_,miss=match(gt,pp,threshold)\n            for j,b in enumerate(gt): draw.rectangle(b[1:].tolist(),outline=\'red\' if j in miss else \'green\',width=3)\n            for b in pp:\n                if b[4]>=threshold: draw.rectangle(b[:4].tolist(),outline=\'cyan\',width=2)\n            im.thumbnail((960,960)); im.save(OUT/f"example-{r[\'id\']}.jpg")\n    summary[\'source_macro_AP\']={k:float(np.mean([summary[c][\'AP\'][k] for c in [\'India\',\'Czech\']])) for k in [\'map50\',\'map50_95\']}\n    save(\'summary.json\',summary)\n    # Verify feature extraction without caching the dataset.\n    hooks=[]; shapes={}; probe=YOLO(checkpoint).model.to(\'cuda\').float().eval()\n    for idx in [16,19]:\n        hooks.append(probe.model[idx].register_forward_hook(lambda m,i,o,idx=idx:shapes.__setitem__(str(idx),list(o.shape))))\n    from ultralytics.data.augment import LetterBox\n    im=np.asarray(Image.open(ROOT/\'prepared/images\'/f"{selected[0][\'id\']}.jpg").convert(\'RGB\'))\n    im=LetterBox(new_shape=(640,640),auto=False)(image=im)\n    tensor=torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))).to(\'cuda\').float()[None]/255\n    try:\n        with torch.no_grad(): probe(tensor)\n    finally:\n        for h in hooks: h.remove()\n    assert shapes[\'16\'][-2:]==[80,80] and shapes[\'19\'][-2:]==[40,40],shapes\n    summary[\'feature_shapes\']=shapes\n    save(\'summary.json\',summary)\n    save(\'storage.json\',dict(output_bytes=sum(p.stat().st_size for p in OUT.rglob(\'*\') if p.is_file())))\n    print(json.dumps({c:{k:v for k,v in m.items() if k in [\'AP\',\'precision\',\'fp_per_image\',\'C\']} for c,m in summary.items() if isinstance(m,dict)},indent=2),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(); parser.add_argument(\'stage\',choices=[\'smoke\',\'train\',\'evaluate\']); args=parser.parse_args()\n    records=setup()\n    try: globals()[args.stage](records)\n    finally:\n        # Inputs remain in the immutable preparation artifact, including on diagnostic failures.\n        shutil.rmtree(ROOT/\'prepared\',ignore_errors=True)\n')


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.192'],check=True)
subprocess.run([sys.executable,'-u','b_run.py',*['evaluate']],check=True)
